In [2]:
# 1. load the tokenizer trained by bpe_tokenization.py
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "bpe_tokenization.py").exists():        # notebook launched from the repo root
    HERE = HERE / "experiments" / "tokenization"
sys.path.insert(0, str(HERE))

from bpe_tokenization import BPETokenizer, render

tok = BPETokenizer.from_pretrained()

print(f"vocab size     : {tok.n_vocab:,}")
print(f"learned merges : {len(tok.merges):,}")
print(f"special tokens : {', '.join(tok.special_tokens)}")
print(f"<|endoftext|>  : {tok.eot_id}     <|pad|> : {tok.pad_id}")
print(f"trained on     : {tok.meta.get('corpus_bytes', 0) / 1e9:.2f} GB")

vocab size     : 8,192
learned merges : 7,930
special tokens : <|endoftext|>, <|pad|>, <|startofinstruction|>, <|endofinstruction|>, <|startofstory|>, <|endofstory|>
<|endoftext|>  : 8186     <|pad|> : 8187
trained on     : 1.11 GB


In [3]:
# 2. encode -> decode on natural English (nothing sampled from the stories)
TEXTS = [
    "The quick brown fox jumps over the lazy dog.",
    "She said, \"I can't believe it's already 5 o'clock!\"",
    "In 2024, researchers published 1,247 papers on transformer architectures.",
    "Backpropagation computes gradients by applying the chain rule in reverse.",
    "Email me at hello@example.com -- or don't; either way, it's fine.",
    "Café, naïve, jalapeño and 日本語 survive byte-for-byte. \U0001f389",
    "Line one.\nLine two follows a newline,\tand a tab.",
    "   leading and trailing whitespace       `` ` ` ` ` ` 1 2465676779&^&^%$  ",
]

print(f"{'ok':<6}{'tokens':>7}{'bytes/tok':>11}   text")
print("-" * 96)

all_ok = True
for text in TEXTS:
    ids = tok.encode(text)
    back = tok.decode(ids)
    ok = back == text
    all_ok = all_ok and ok
    print(f"{'PASS' if ok else 'FAIL':<6}{len(ids):>7}{len(text.encode()) / len(ids):>11.2f}   {text!r}")

print("-" * 96)
print("all strings decoded back to the exact original" if all_ok else "MISMATCH FOUND")

ok     tokens  bytes/tok   text
------------------------------------------------------------------------------------------------
PASS       10       4.40   'The quick brown fox jumps over the lazy dog.'
PASS       18       2.83   'She said, "I can\'t believe it\'s already 5 o\'clock!"'
PASS       31       2.35   'In 2024, researchers published 1,247 papers on transformer architectures.'
PASS       22       3.32   'Backpropagation computes gradients by applying the chain rule in reverse.'
PASS       26       2.50   "Email me at hello@example.com -- or don't; either way, it's fine."
PASS       42       1.57   'Café, naïve, jalapeño and 日本語 survive byte-for-byte. 🎉'
PASS       17       2.82   'Line one.\nLine two follows a newline,\tand a tab.'
PASS       50       1.48   '   leading and trailing whitespace       `` ` ` ` ` ` 1 2465676779&^&^%$  '
------------------------------------------------------------------------------------------------
all strings decoded back to the exact original


In [4]:
# 3. your own text: encode -> ids -> decode -> compare
DEFAULT = "Tokenizers turn text into integers, and integers back into text."

try:
    TEXT = "Text to tokenize (press Enter for the default): ".strip()
except (EOFError, OSError):
    TEXT = ""
TEXT = TEXT or DEFAULT

ids = tok.encode(TEXT)
back = tok.decode(ids)

print(f"\ninput   : {TEXT!r}")
print(f"bytes   : {len(TEXT.encode())}")
print(f"tokens  : {len(ids)}   ({len(TEXT.encode()) / len(ids):.2f} bytes/token)")
print(f"ids     : {ids}")

print("\npiece by piece   (Ġ = space, Ċ = newline):")
for i in ids[:40]:
    print(f"  {i:>6}  {render(tok.vocab[i])}")
if len(ids) > 40:
    print(f"  ... {len(ids) - 40} more")

print(f"\ndecoded : {back!r}")
print(f"\nidentical to input: {back == TEXT}")


input   : 'Text to tokenize (press Enter for the default):'
bytes   : 47
tokens  : 20   (2.35 bytes/token)
ids     : [84, 4390, 268, 268, 1295, 1798, 32, 40, 112, 1069, 852, 362, 284, 376, 262, 6639, 97, 2092, 41, 58]

piece by piece   (Ġ = space, Ċ = newline):
      84  T
    4390  ext
     268  Ġto
     268  Ġto
    1295  ken
    1798  ize
      32  Ġ
      40  (
     112  p
    1069  ress
     852  ĠE
     362  nt
     284  er
     376  Ġfor
     262  Ġthe
    6639  Ġdef
      97  a
    2092  ult
      41  )
      58  :

decoded : 'Text to tokenize (press Enter for the default):'

identical to input: True


In [ ]:
# 4. corpus -> one flat uint16 token stream on disk (.npy)
import multiprocessing as mp
import time
from itertools import islice

import numpy as np

# Python 3.14 starts workers with `spawn` on macOS, and a spawned worker
# re-imports __main__ -- which in a notebook is not a file, so the Pool inside
# encode_batch() would die on import. `fork` inherits the already-built
# tokenizer instead: correct here, and free (no re-import).
if mp.get_start_method(allow_none=True) != "fork":
    mp.set_start_method("fork", force=True)

from bpe_tokenization import N_CORES, TRAIN_TXT, VAL_TXT, prepare_line

DTYPE = np.uint16          # 8,192 ids fit in 2 bytes; halves the file and the read
assert tok.n_vocab <= np.iinfo(DTYPE).max, f"vocab {tok.n_vocab} no longer fits {DTYPE}"


def tokenize_split(txt_path, out_path, max_records=None, shard_records=20_000,
                   workers=N_CORES, add_eot=True):
    """Encode a whole split into one flat uint16 array and np.save it.

    Layout: every record's ids end to end, `<|endoftext|>` after each. No
    padding, no per-record offsets -- the training loader memmaps the file and
    slices a block of any length at a random offset, and the separator token is
    what tells the model one story ended and another began.

    The corpus is read in shards so only `shard_records` stories exist as
    Python strings at once, and the shards are copied into one preallocated
    array at the end (rather than np.concatenate, which would transiently hold
    two full copies of a 1 GB corpus).
    """
    txt_path, out_path = Path(txt_path), Path(out_path)
    eot = tok.eot_id
    shards, n_tokens, n_records, n_bytes = [], 0, 0, 0
    t0 = time.time()

    def flush(lines):
        nonlocal n_tokens, n_records, n_bytes
        if not lines:
            return
        ids = []
        for rec in tok.encode_batch(lines, workers=workers):
            ids.extend(rec)
            if add_eot:
                ids.append(eot)
        shards.append(np.asarray(ids, dtype=DTYPE))
        n_records += len(lines)
        n_tokens += len(ids)
        n_bytes += sum(len(t.encode()) for t in lines)
        dt = time.time() - t0
        print(f"  {n_records:>9,} records  {n_tokens:>13,} tokens  "
              f"{n_bytes / 1e6:8.1f} MB read  {dt:6.1f}s  "
              f"{n_tokens / max(dt, 1e-9):>10,.0f} tok/s", end="\r", flush=True)

    print(f"encoding {txt_path.name}  ({txt_path.stat().st_size / 1e9:.2f} GB) "
          f"on {workers} processes")
    buf = []
    with txt_path.open(encoding="utf-8") as f:
        for line in islice(f, max_records):
            buf.append(prepare_line(line.rstrip("\n")))
            if len(buf) >= shard_records:
                flush(buf)
                buf = []
        flush(buf)
    print()

    arr = np.empty(n_tokens, dtype=DTYPE)
    at = 0
    while shards:                              # pop as we copy -> shard memory freed
        s = shards.pop(0)
        arr[at:at + len(s)] = s
        at += len(s)
    assert at == n_tokens

    np.save(out_path, arr)
    dt = time.time() - t0
    stats = {
        "path": str(out_path),
        "records": n_records,
        "tokens": n_tokens,
        "bytes": n_bytes,
        "bytes_per_token": n_bytes / max(n_tokens, 1),
        "tokens_per_record": n_tokens / max(n_records, 1),
        "seconds": dt,
    }
    print(f"  saved -> {out_path}")
    print(f"  {n_tokens:,} tokens  {out_path.stat().st_size / 1e6:,.1f} MB on disk  "
          f"{stats['bytes_per_token']:.2f} bytes/token  "
          f"{stats['tokens_per_record']:.0f} tokens/record  {dt:.1f}s")
    del arr
    return stats

In [ ]:
# 5. validation split first -- 12 MB, a couple of seconds, proves the whole path
OUT_DIR = VAL_TXT.parent           # Datasets/, already gitignored (*.npy is too)

val_stats = tokenize_split(VAL_TXT, OUT_DIR / "tinystories_instruct_val_uint16.npy")

In [ ]:
# 6. the full 1.1 GB training split (the long cell -- minutes, not seconds)
train_stats = tokenize_split(TRAIN_TXT, OUT_DIR / "tinystories_instruct_train_uint16.npy")

In [ ]:
# 7. read the .npy back the way the trainer will, and check it survived the trip
def verify(stats, txt_path, chunk=50_000_000):
    path = Path(stats["path"])
    arr = np.load(path, mmap_mode="r")          # mmap: never loads 800 MB into RAM

    # one streamed pass for the two things worth knowing: no id escaped the
    # vocab, and the document count matches the record count
    hi, n_eot = 0, 0
    for i in range(0, arr.size, chunk):
        block = np.asarray(arr[i:i + chunk])
        hi = max(hi, int(block.max()))
        n_eot += int((block == tok.eot_id).sum())

    # first record: slice up to the first separator and decode it back
    head = np.asarray(arr[:8192])
    end = int(np.flatnonzero(head == tok.eot_id)[0])
    with txt_path.open(encoding="utf-8") as f:
        original = prepare_line(next(f).rstrip("\n"))
    decoded = tok.decode(head[:end].tolist())

    print(f"\n{path.name}")
    print(f"  dtype / shape   : {arr.dtype}  {arr.shape}")
    print(f"  on disk         : {path.stat().st_size / 1e6:,.1f} MB")
    print(f"  max token id    : {hi}  (vocab {tok.n_vocab}, in range: {hi < tok.n_vocab})")
    print(f"  <|endoftext|>   : {n_eot:,}  (records encoded: {stats['records']:,}, "
          f"match: {n_eot == stats['records']})")
    print(f"  first record    : {end} tokens, round-trip "
          f"{'exact' if decoded == original else 'MISMATCH'}")
    print(f"  preview         : {decoded[:200]!r} ...")

    # and the way a dataloader actually uses it: a random block of block_size+1
    block_size = 512
    rng = np.random.default_rng(0)
    at = int(rng.integers(0, arr.size - block_size - 1))
    batch = np.asarray(arr[at:at + block_size + 1]).astype(np.int64)
    print(f"  sample batch    : x={batch[:-1].shape} y={batch[1:].shape} "
          f"from offset {at:,}")


verify(val_stats, VAL_TXT)
verify(train_stats, TRAIN_TXT)